In [1]:
import os
import shutil
import random

# Define your source and destination paths
source_dir = 'out'  # Assuming the 'out' folder is in your current working directory
train_dir = 'data/train'
test_dir = 'data/test'
val_split = 0.2  # 20% for validation/testing

# Delete everything in the data folder before creating new directories
if os.path.exists('data'):
    shutil.rmtree('data')

# Ensure the target directories exist
os.makedirs(f'{train_dir}/mii_images', exist_ok=True)
os.makedirs(f'{train_dir}/not_mii_images', exist_ok=True)
os.makedirs(f'{test_dir}/mii_images', exist_ok=True)
os.makedirs(f'{test_dir}/not_mii_images', exist_ok=True)

# Function to split and copy the images
def split_data(source_folder, dest_train_folder, dest_test_folder, split_ratio):
    # Get list of all files in the source folder
    all_images = os.listdir(source_folder)
    random.shuffle(all_images)  # Shuffle to randomize the selection
    
    # Calculate split index
    split_index = int(len(all_images) * (1 - split_ratio))
    
    # Train data (copying to the train folder)
    train_images = all_images[:split_index]
    for image in train_images:
        shutil.copy(os.path.join(source_folder, image), os.path.join(dest_train_folder, image))
    
    # Test/Validation data (copying to the test folder)
    test_images = all_images[split_index:]
    for image in test_images:
        shutil.copy(os.path.join(source_folder, image), os.path.join(dest_test_folder, image))

# Split mii_images
split_data(os.path.join(source_dir, 'mii_images'), 
           os.path.join(train_dir, 'mii_images'), 
           os.path.join(test_dir, 'mii_images'), 
           val_split)

# Split not_mii_images
split_data(os.path.join(source_dir, 'not_mii_images'), 
           os.path.join(train_dir, 'not_mii_images'), 
           os.path.join(test_dir, 'not_mii_images'), 
           val_split)

print("Data split complete.")

Data split complete.


In [9]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [10]:
# Define the stronger CNN model
def create_mii_classifier_model_7():
    model = Sequential()
    
    # First convolutional block
    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(50, 50, 3)))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))
    
    # Second convolutional block
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))
    
    # Third convolutional block
    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))
    
    # Fourth convolutional block (optional, depending on dataset complexity)
    model.add(Conv2D(256, (3, 3), activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))
    
    # Global Average Pooling instead of Flatten
    model.add(GlobalAveragePooling2D())
    
    # Fully connected layer
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.5))  # Dropout for regularization
    
    # Output layer (binary classification)
    model.add(Dense(1, activation='sigmoid'))
    
    # Compile the model with a lower learning rate
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])
    
    return model

# Create the new model
mii_classifier_model_7 = create_mii_classifier_model_7()

# Summarize the model architecture
mii_classifier_model_7.summary()

/Users/qayyuma/Documents/GitHub/wii-project/wii/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 48, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 22, 22, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 22, 22, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 11, 11, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 9, 9, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 9, 9, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 2, 2, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 2, 2, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 1, 1, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 456,385 (1.74 MB)

 Trainable params: 455,425 (1.74 MB)

 Non-trainable params: 960 (3.75 KB)

In [11]:
# Assuming you have your data stored in directories as 'data/train/mii_images', 'data/train/not_mii_images', 'data/test/mii_images', and 'data/test/not_mii_images'
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

# Load training and validation data
train_data = train_datagen.flow_from_directory(
    'data/train',  # directory with 'mii_images' and 'not_mii_images' subdirectories
    target_size=(50, 50),
    batch_size=32,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    'data/test',
    target_size=(50, 50),
    batch_size=32,
    class_mode='binary'
)

# Adding callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001)

# Train the new model
history = mii_classifier_model_7.fit(
    train_data,
    steps_per_epoch=len(train_data),
    epochs=50,  # Increased epochs for better performance
    validation_data=val_data,
    validation_steps=len(val_data),
    callbacks=[early_stopping, reduce_lr]
)

Found 37184 images belonging to 2 classes.
Found 9297 images belonging to 2 classes.
Epoch 1/50


/Users/qayyuma/Documents/GitHub/wii-project/wii/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1162/1162 ━━━━━━━━━━━━━━━━━━━━ 53s 44ms/step - accuracy: 0.9521 - loss: 0.1282 - val_accuracy: 0.9985 - val_loss: 0.0052 - learning_rate: 1.0000e-04
Epoch 2/50
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 0s 22us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - learning_rate: 1.0000e-04
Epoch 3/50
   1/1162 ━━━━━━━━━━━━━━━━━━━━ 1:42 89ms/step - accuracy: 1.0000 - loss: 0.0031

2024-11-11 10:55:00.054847: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
/opt/homebrew/Caskroom/miniforge/base/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)
2024-11-11 10:55:00.063532: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
/Users/qayyuma/Documents/GitHub/wii-project/wii/lib/python3.10/site-packages/keras/src/callbacks/early_stopping.py:155: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/

1162/1162 ━━━━━━━━━━━━━━━━━━━━ 51s 44ms/step - accuracy: 0.9987 - loss: 0.0061 - val_accuracy: 0.9996 - val_loss: 0.0015 - learning_rate: 1.0000e-04
Epoch 4/50
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 0s 9us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - learning_rate: 1.0000e-04
Epoch 5/50
   3/1162 ━━━━━━━━━━━━━━━━━━━━ 47s 41ms/step - accuracy: 1.0000 - loss: 8.1872e-04 

2024-11-11 10:55:50.976237: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]


1162/1162 ━━━━━━━━━━━━━━━━━━━━ 50s 43ms/step - accuracy: 0.9989 - loss: 0.0043 - val_accuracy: 0.9973 - val_loss: 0.0070 - learning_rate: 1.0000e-04
Epoch 6/50
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 0s 8us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - learning_rate: 1.0000e-04
Epoch 7/50
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 50s 43ms/step - accuracy: 0.9988 - loss: 0.0041 - val_accuracy: 0.9999 - val_loss: 3.2725e-04 - learning_rate: 1.0000e-04
Epoch 8/50
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 0s 9us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - learning_rate: 1.0000e-04
Epoch 9/50
   3/1162 ━━━━━━━━━━━━━━━━━━━━ 44s 38ms/step - accuracy: 1.0000 - loss: 8.0701e-05 

2024-11-11 10:57:30.881596: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]


 228/1162 ━━━━━━━━━━━━━━━━━━━━ 39s 42ms/step - accuracy: 0.9999 - loss: 4.9887e-04

KeyboardInterrupt: 

In [ ]:
mii_classifier_model_7.save('mii_classifier_model_7.keras')

NameError: name 'mii_classifier_model_6' is not defined

In [12]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='upper left')
plt.show()

# Plot training & validation loss values
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper left')
plt.show()

NameError: name 'history' is not defined

In [ ]:
# Assuming you have a new dataset of test images
test_data = val_datagen.flow_from_directory(
    'data/test',  # directory containing test images
    target_size=(50, 50),
    batch_size=32,
    class_mode='binary'

# Evaluate the model
test_loss, test_accuracy = mii_classifier_model_6.evaluate(test_data)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")


Found 9297 images belonging to 2 classes.


NameError: name 'mii_classifier_model_6' is not defined

Model 4's Test Accuracy: 99.58%
Model 5's Test Accuracy: 99.93%
Model 6's Test Accuracy: 99.98%

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

# Get the true labels and predicted labels for the test set
# Predict probabilities and convert them to binary predictions (0 or 1)
y_true = test_data.classes
y_pred_prob = mii_classifier_model_6.predict(test_data)
y_pred = (y_pred_prob > 0.5).astype("int32").flatten()

# Generate the confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

# Calculate additional metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
misclassification_rate = 1 - accuracy  # Alternatively, (FP + FN) / Total

# Print metrics
print("\nMetrics:")
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")
print(f"Misclassification Rate: {misclassification_rate:.2f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=test_data.class_indices.keys()))


ModuleNotFoundError: No module named 'seaborn'